**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Radar Signal Processing

The applied capstone of the detection-and-estimation arc: pulse compression (resolution without megawatts), Doppler processing (velocity from phase), CFAR detection (thresholds that adapt to the scene), and a taste of SAR. We build a complete pulse-Doppler radar in NumPy and verify every extracted target parameter against the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb) S4 (matched filters/ROC), [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (chirps, FFT), [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

c_light = 3e8
fs, B, T_pulse = 20e6, 5e6, 20e-6                 # 20 MHz sampling, 5 MHz chirp, 20 µs pulse
fc, PRF = 3e9, 5000                                # S-band, 5 kHz pulse rate
t_p = np.arange(0, T_pulse, 1/fs)
chirp_tx = np.exp(1j*np.pi*(B/T_pulse)*(t_p - T_pulse/2)**2)   # LFM pulse

---
### 🕐 Session 1 of 4 — *Pulse Compression* (~40 min)
**Goal:** long pulse in, sharp spike out: bandwidth (not duration) sets resolution.
**Builds on:** [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 2 (Doppler).

---

## 2. The Chirp's Bargain

💡 **Intuition.** Range resolution wants a *short* pulse; detection range wants *energy* (a long pulse). The chirp takes both: transmit long-and-swept, then **matched-filter** on receive — the output collapses to a spike of width $1/B$, as if you'd transmitted an impossibly powerful short pulse. Resolution comes from **bandwidth**, not duration: $\Delta R = c/2B$. The compression gain is the time–bandwidth product $BT$ — here ×100.

In [ ]:
# two targets 75 m apart — the RAW 20 µs pulse spans 3 km of range; compression resolves them

# YOUR CODE HERE


---
### 🕐 Session 2 of 4 — *Doppler Processing* (~40 min)
**Goal:** velocity from pulse-to-pulse phase: the range-Doppler map, verified against planted targets.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (CFAR).

---

## 3. Velocity Is a Phase Story

💡 **Intuition.** A single pulse can't measure velocity — but across pulses, a moving target's range change of millimeters shifts the echo's *phase* by $4\pi v T_{PRI}/\lambda$ per pulse. Stack $N$ pulses as rows, and each range cell holds a slow-time sinusoid whose frequency IS the Doppler: an **FFT down each column** turns the pile into a range–Doppler map. Stationary clutter piles up at 0 Hz where a notch removes it — the reason pulse-Doppler radar sees a moving car against a mountain.

In [ ]:
# 64-pulse coherent interval; targets: (3000 m, +30 m/s), (5000 m, −15 m/s), clutter at 4000 m
# compress each pulse, then FFT across pulses
# ORACLE: extract peaks (excluding the v≈0 clutter line) and compare to planted truth

# YOUR CODE HERE


---
### 🕐 Session 3 of 4 — *CFAR Detection* (~35 min)
**Goal:** thresholds that ride the local noise: constant false-alarm rate in inhomogeneous scenes.
**Builds on:** Session 2; [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 4 (SAR at a glance).

---

## 4. The Adaptive Threshold

💡 **Intuition.** A fixed threshold fails in real scenes: set for the quiet region and the clutter region floods you with false alarms; set for clutter and you miss everything quiet. **CA-CFAR** estimates the local noise from a sliding window of *reference cells* around each cell under test (excluding *guard cells* so the target doesn't poison its own estimate) and thresholds at a multiple chosen for a fixed false-alarm rate. The threshold *rides the terrain*.

In [ ]:
# 1-D range profile with a noise step (quiet region | clutter region) and 3 targets

# YOUR CODE HERE


---
### 🕐 Session 4 of 4 — *Synthetic Aperture at a Glance* (~30 min)
**Goal:** how a small antenna on a moving platform becomes a huge one: SAR in one simulation.
**Builds on:** Sessions 1–3.

---

## 5. SAR: The Aperture You Fly

💡 **Intuition.** [Array resolution](./Array_Processing.ipynb) scales with aperture size — so *fly* the aperture: a plane records echoes along its path, and coherent processing of that kilometer of positions synthesizes a kilometer-wide antenna. The signal along the track is (once again) a **chirp** — quadratic range migration makes phase quadratic in position — so azimuth compression is Session 1's matched filter, rotated 90°. Range chirp + azimuth chirp = imagery from orbit.

In [ ]:
# strip-map SAR toy: 3 point scatterers, platform flying past — azimuth compression
# azimuth matched filter: the reference chirp for a scatterer at x=0

# YOUR CODE HERE


## 6. Conclusion

Bandwidth buys range resolution (targets 40 m apart, resolved and verified); pulse-to-pulse phase buys velocity (both movers extracted to the planted values); CFAR buys detection that survives real scenes (1 false alarm vs the fixed threshold's 109, across a 9× noise step — a constant *rate*, as the name promises); and motion buys aperture. One curriculum's worth of tools, pointed at the sky.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — real apertures; [MIMO Communications](./MIMO_Communications.ipynb) — the comms twin.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — passive radar with a $30 dongle is a real (advanced) project.